In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df_bronze_taxi = spark.table("databrickstask.bronze.taxi")
df_bronze_drivers = spark.table("databrickstask.bronze.drivers")
df_bronze_users = spark.table("databrickstask.bronze.users")
window_spec = Window.orderBy("name", "car_number")
df_silver_drivers = spark.table("databrickstask.silver.drivers")
drivers_count = df_silver_drivers.count()

df_silver_taxi = (
    df_bronze_taxi
    .filter((F.col("passenger_count") > 0) &
            (F.col("trip_distance") > 0) &
            (F.col("fare_amount") > 0 ))
    .dropDuplicates()   
    .withColumn(
        "description",
        F.when(F.col("payment_type") == 1, "Credit Card")
        .when(F.col("payment_type") == 2, "Cash")
        .otherwise("Other"))
    
    .withColumn("ingest_time", F.current_timestamp())
    .withColumn("driver_id", F.pmod(F.hash("PULocationID", "tpep_pickup_datetime", "tpep_dropoff_datetime"), F.lit(drivers_count)) + 1)
)

df_silver_drivers = (
    df_bronze_drivers
    .filter((F.col("rating").between(1.0, 5.0))&
            (F.col("experience") > 0)
             ) 
    .dropDuplicates(["name", "car_number"])
    .withColumn("ingest_time", F.current_timestamp())
    .withColumn("driver_id", F.row_number().over(window_spec))
)

df_silver_users = (
    df_bronze_users
    .filter(F.col("name").isNotNull())
    .dropDuplicates()
    .withColumn("ingest_time", F.current_timestamp())
    .withColumn("registration_date", F.current_date())
)


(
    df_silver_taxi.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databrickstask.silver.taxi")
)
(
    df_silver_drivers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databrickstask.silver.drivers")
)
(
    df_silver_users.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databrickstask.silver.users")
)
